In [11]:
# scripts/05_estimate_spy_sector_weights.py
"""
Estimate SPY sector weights monthly by regressing SPY on sector ETF returns
using a robust NNLS (non-negative least squares) with fallbacks.

Outputs:
  data/processed/spy_sector_weights_monthly.parquet
  data/processed/spy_sector_weights_daily.parquet

Why NNLS?
- Tránh lỗi SVD không hội tụ của OLS khi thiếu số liệu/đa cộng tuyến
- Bảo đảm w >= 0 và chuẩn hóa sum(w) = 1
"""

from pathlib import Path
import numpy as np
import pandas as pd
from scipy.optimize import nnls   # <- robust solver

RET_SPY = Path("data/processed/benchmark_ret.parquet")
RET_SEC = Path("data/raw/sector_etf_returns.parquet")
OUTDIR  = Path("data/processed")
OUTDIR.mkdir(parents=True, exist_ok=True)

# --- Params ---
MIN_ROWS = 8        # tối thiểu số phiên trong THÁNG TRƯỚC để ước lượng
RIDGE    = 1e-6     # fallback ridge nhỏ nếu nnls không dùng được (hiếm)
EQ_FALLBACK = True  # fallback equal-weight khi thất bại

# --- Load ---
# --- Load benchmark & sector returns (FIX) ---
tmp_spy = pd.read_parquet(RET_SPY)  # có thể là Series hoặc DataFrame 1 cột
if isinstance(tmp_spy, pd.Series):
    spy = tmp_spy.copy()
    if spy.name is None:
        spy.name = "SPY"
elif isinstance(tmp_spy, pd.DataFrame):
    if "SPY" in tmp_spy.columns:
        spy = tmp_spy["SPY"].copy()
        spy.name = "SPY"
    elif tmp_spy.shape[1] == 1:
        spy = tmp_spy.iloc[:, 0].copy()
        spy.name = "SPY"
    else:
        raise RuntimeError(
            f"benchmark_ret.parquet có nhiều cột ({list(tmp_spy.columns)}) mà không có 'SPY'."
        )
else:
    raise RuntimeError("RET_SPY không phải Series/DataFrame hợp lệ.")

# Đồng bộ index và sắp xếp
common = spy.index.intersection(sec.index)
spy = spy.reindex(common).sort_index()
sec = sec.reindex(common).sort_index().ffill()

sector_cols = list(sec.columns)

# --- Helper: ước lượng trọng số cho một tháng m bằng dữ liệu tháng m-1
def estimate_month_weight(prev_mask: pd.Series) -> np.ndarray:
    X = sec.loc[prev_mask, sector_cols].copy()
    y = spy.loc[prev_mask].copy()

    # Làm sạch: drop hàng NaN, drop cột có toàn NaN hoặc phương sai ~ 0
    df = X.copy()
    df["y"] = y
    df = df.dropna(axis=0, how="any")
    if df.shape[0] < MIN_ROWS:
        return None

    Xc = df[sector_cols].copy()
    yc = df["y"].values

    # loại cột phẳng (std~0) trong cửa sổ
    std = Xc.std(axis=0).replace(0, np.nan)
    valid_cols = std[std.notna()].index.tolist()
    if len(valid_cols) == 0:
        return None

    Xv = Xc[valid_cols].values
    # NNLS
    try:
        w_part, _ = nnls(Xv, yc)
        # map về full vector
        w_full = np.zeros(len(sector_cols), dtype=float)
        for i, col in enumerate(valid_cols):
            j = sector_cols.index(col)
            w_full[j] = w_part[i]
    except Exception:
        # Fallback ridge nhỏ: (X'X + λI)w = X'y (trên valid_cols)
        try:
            XtX = Xv.T @ Xv
            XtX += RIDGE * np.eye(XtX.shape[0])
            Xty = Xv.T @ yc
            w_part = np.linalg.solve(XtX, Xty)
            w_part = np.clip(w_part, 0.0, None)
            w_full = np.zeros(len(sector_cols), dtype=float)
            for i, col in enumerate(valid_cols):
                j = sector_cols.index(col)
                w_full[j] = w_part[i]
        except Exception:
            return None

    s = w_full.sum()
    if s <= 0:
        return None
    return w_full / s

# --- Vòng lặp theo tháng ---
months = pd.period_range(spy.index.min().to_period('M'),
                         spy.index.max().to_period('M'), freq='M')

rows = []
idxs = []
last_w = np.ones(len(sector_cols)) / len(sector_cols)  # khởi tạo equal-weight

for m in months:
    prev = m - 1
    prev_mask = (spy.index.to_period('M') == prev)

    w = estimate_month_weight(prev_mask)
    if w is None:
        # fallback: dùng trọng số tháng trước nếu có, hoặc equal-weight
        w = last_w.copy() if not np.allclose(last_w.sum(), 0) else (np.ones(len(sector_cols))/len(sector_cols))

    last_w = w.copy()
    rows.append(w)
    idxs.append(m.to_timestamp('M'))  # mốc cuối tháng

wb_monthly = pd.DataFrame(rows, index=pd.Index(idxs, name="date"), columns=sector_cols).sort_index()

# Daily: FFill lên lịch giao dịch của sector returns
wb_daily = wb_monthly.reindex(sec.index, method="ffill").fillna(0.0)

# --- Save ---
wb_monthly.to_parquet(OUTDIR / "spy_sector_weights_monthly.parquet")
wb_daily.to_parquet(OUTDIR / "spy_sector_weights_daily.parquet")

print("Saved:", (OUTDIR / "spy_sector_weights_monthly.parquet").resolve())
print("Saved:", (OUTDIR / "spy_sector_weights_daily.parquet").resolve())

# --- Kiểm tra nhanh ---
assert not wb_monthly.isna().any().any(), "Monthly weights contain NaN!"
assert np.allclose(wb_monthly.sum(axis=1).values, 1.0, atol=1e-6), "Monthly weights not summing to 1."
assert not wb_daily.isna().any().any(), "Daily weights contain NaN!"

Saved: C:\Users\ACER\python\Dissertation\Port Dashboard\data\processed\spy_sector_weights_monthly.parquet
Saved: C:\Users\ACER\python\Dissertation\Port Dashboard\data\processed\spy_sector_weights_daily.parquet


In [13]:
tmp = pd.read_parquet("data/processed/benchmark_ret.parquet")
print(type(tmp), getattr(tmp, "shape", None), getattr(tmp, "name", None))

<class 'pandas.core.frame.DataFrame'> (1825, 1) None
